In [3]:
import pandas as pd
import duckdb
import datania

# Load datasets
household_df = datania.generate_household_survey(n_households=1000, survey_year=2025, seed=77)
provinces_df = datania.get_provinces()
districts_df = datania.get_districts()

# Task 1: Examine datasets
print("=== HOUSEHOLD DATA ===")
print(household_df.head())
print(f"\nColumns: {household_df.columns.tolist()}")

print("\n=== PROVINCES REFERENCE ===")
print(provinces_df)


=== HOUSEHOLD DATA ===
            hh_id province            district urban_rural  hh_size  \
0  DHHS2025-00001      LKS   Estimate District       Rural        3   
1  DHHS2025-00002      STH    Frequency Plains       Urban        4   
2  DHHS2025-00003      STH       Mode District       Rural        4   
3  DHHS2025-00004      LKS  Parameter District       Rural        5   
4  DHHS2025-00005      LKS  Parameter District       Urban        7   

   n_children  n_working_age  n_elderly  monthly_income  monthly_expenditure  \
0           2              1          0            7289                 6947   
1           0              4          0           98374                74847   
2           2              2          0           63671                44854   
3           2              3          0           15311                13917   
4           3              4          0           22461                17279   

   has_electricity  has_improved_water  has_improved_sanitation  owns

In [4]:

# Task 2: Left join to add province info
# Hint: The common key is 'province' in household_df and 'province_code' in provinces_df
# YOUR CODE HERE
merged_df = pd.merge(
    household_df,
    provinces_df,
    left_on="province",
    right_on="province_code",
    how="left"
)

print(f"\nMerged shape: {merged_df.shape}")



Merged shape: (1000, 25)


In [5]:

# Task 3: Check for unmatched records
unmatched = merged_df[merged_df["province_name"].isna()]
print(f"Unmatched records: {len(unmatched)}")


Unmatched records: 0


In [6]:

# Task 4: Create income per capita
# YOUR CODE HERE

# Task 5: Use DuckDB SQL to calculate average income by province
print("\n=== AVERAGE INCOME BY PROVINCE (SQL) ===")
query = """
    SELECT province, province_name, AVG(monthly_income) as avg_income
    FROM merged_df
    GROUP BY province, province_name
    ORDER BY avg_income DESC
"""
# YOUR CODE HERE
result = duckdb.query(query).to_df()
print(result)

# Task 6: Recode household size into categories
# YOUR CODE HERE
merged_df["hh_size_category"] = pd.cut(
    merged_df["hh_size"],
    bins=[0, 2, 5, 100],
    labels=["Small", "Medium", "Large"]
)

# Verify recoding
print("\n=== HOUSEHOLD SIZE CATEGORIES ===")
print(merged_df["hh_size_category"].value_counts())

# Task 7: Save to CSV
# YOUR CODE HERE
merged_df.to_csv("households_merged_2025.csv", index=False)
print("\nData saved to households_merged_2025.csv")


=== AVERAGE INCOME BY PROVINCE (SQL) ===
  province      province_name    avg_income
0      CTR   Central Province  66434.478417
1      STH  Southern Province  57658.840708
2      EST   Eastern Province  44827.630137
3      LKS  Lakeside Province  37720.993333
4      NTH  Northern Province  33727.038835
5      WST   Western Province  27533.474227

=== HOUSEHOLD SIZE CATEGORIES ===
hh_size_category
Medium    542
Large     382
Small      76
Name: count, dtype: int64

Data saved to households_merged_2025.csv
